# 研发费用加计扣除政策、研发投入调整与企业创新效率
## ——基于制造业政策冲击的上市公司经验证据

**V6 效率分析 (Jupyter Notebook 版本)**

本 notebook 完整复现 V6 分析流程：

1. 数据审计
2. 变量预处理
3. 基准创新数量模型
4. 研发投入行为模型
5. 创新效率主模型
6. 效率拆解
7. 单位敏感性
8. 替代效率指标
9. 事件研究
10. 安慰剂检验
11. 异质性分析
12. 图表生成


## 1. 环境设置

In [ ]:
import os, sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Matplotlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
import seaborn as sns

# Chinese font
plt.rcParams['font.family'] = 'Noto Sans CJK SC'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# linearmodels
from linearmodels.panel import PanelOLS

# Paths
PROJ = Path("/home/u2023312303/裴的实验/tec_expenditure")
OUT = PROJ / "v6/outputs"
FIG = OUT / "figures"
TBL = OUT / "tables"
SCR = OUT / "scripts"
for d in [OUT, FIG, TBL, SCR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete.")
print(f"Noto Sans CJK SC available: {any('Noto Sans CJK SC' == f.name for f in fm.fontManager.ttflist)}")


## 2. 数据读取与审计

In [ ]:
# Read v5 preprocessed panel
df = pd.read_csv(PROJ / "v5/data/v5_clean_panel.csv")
df["stock_code"] = df["stock_code"].astype(str).str.zfill(6)
df["year"] = df["year"].astype(int)
df = df[df["year"] <= 2024].copy()

print(f"Raw: {len(df):,} obs × {df['stock_code'].nunique():,} firms, year={df['year'].min()}-{df['year'].max()}")

# Duplicate check
dup_all = df.duplicated(subset=["stock_code", "year"]).sum()
print(f"stock_code-year duplicates: {dup_all}")

# Sub-samples
samples = {}
for yr_s, yr_e, label in [(2017, 2022, "2017-2022"), (2017, 2024, "2017-2024"),
                            (2017, 2020, "2017-2020"), (2016, 2022, "2016-2022")]:
    s = df[df["year"].between(yr_s, yr_e)]
    dup = s.duplicated(subset=["stock_code", "year"]).sum()
    max_o = s.groupby("stock_code").size().max()
    over = (s.groupby("stock_code").size() > (yr_e - yr_s + 1)).sum()
    print(f"  {label}: {len(s):,} obs × {s['stock_code'].nunique():,} firms, dup={dup}, max_obs={max_o}, over={over}")
    samples[label] = s

assert dup_all == 0, f"FATAL: {dup_all} duplicates!"
print("\n>>> 最终面板为唯一企业年度面板，不存在 stock_code-year 重复观测。 <<<")


## 3. 变量预处理

In [ ]:
# Ratio conversion
ratio_checks = {
    "rd_intensity": "研发强度",
    "rd_staff_ratio": "研发人员占比",
    "province_sci_tech_ratio": "省财政科技支出占比",
    "province_rd_intensity": "省R&D强度",
}
for v, desc in ratio_checks.items():
    if v in df.columns:
        col = pd.to_numeric(df[v], errors="coerce")
        med = col.median()
        if med > 1:
            df[v] = col / 100.0
            print(f"  {v}: median {med:.4g} → {df[v].median():.4g} (/100)")

# Unit-variant rd_expense
df["rd_expense_yuan"] = pd.to_numeric(df["rd_expense"], errors="coerce").clip(lower=0)
df["rd_expense_10k"] = df["rd_expense_yuan"] / 10000
df["rd_expense_million"] = df["rd_expense_yuan"] / 1000000
for label, col in [("yuan", "rd_expense_yuan"), ("10k", "rd_expense_10k"), ("million", "rd_expense_million")]:
    df[f"ln_rd_expense_{label}"] = np.log1p(df[col])

# Log patents
for v in ["invention_apply", "invention_grant", "patent_apply_total", "patent_grant_total"]:
    df[v] = pd.to_numeric(df[v], errors="coerce").fillna(0).clip(lower=0)
    df[f"ln_{v}"] = np.log1p(df[v])

# Log staff
df["rd_staff"] = pd.to_numeric(df["rd_staff"], errors="coerce").fillna(0).clip(lower=0)
df["ln_rd_staff"] = np.log1p(df["rd_staff"])

# Innovation efficiency (log-difference)
for dv in ["ln_invention_apply", "ln_invention_grant"]:
    short = dv.replace("ln_", "")
    for unit, ulabel in [("yuan", "yuan"), ("10k", "10k"), ("million", "million")]:
        rd_var = f"ln_rd_expense_{ulabel}"
        if rd_var in df.columns:
            df[f"eff_{short}_rd_{unit}"] = df[dv] - df[rd_var]
    df[f"eff_{short}_staff"] = df[dv] - df["ln_rd_staff"]

# Ratio-type efficiency
for num, nlabel in [("invention_apply", "apply"), ("invention_grant", "grant")]:
    for den_col, dlabel in [("rd_expense_yuan", "rd"), ("rd_staff", "staff")]:
        ratio_name = f"{nlabel}_per_{dlabel}"
        df[ratio_name] = np.where(df[den_col] > 0, df[num] / df[den_col], np.nan)
        df[f"asinh_{ratio_name}"] = np.arcsinh(df[ratio_name])

# Winsorize
WINSOR = ["ln_assets", "roa", "cashflow_ratio", "firm_age", "rd_intensity",
          "rd_staff_ratio", "ln_rd_expense_yuan", "ln_rd_expense_10k",
          "ln_rd_expense_million", "ln_rd_staff", "ln_invention_apply",
          "ln_invention_grant", "ln_patent_apply_total", "ln_patent_grant_total"]
for c in df.columns:
    if c.startswith("eff_") or c.startswith("asinh_"):
        WINSOR.append(c)

for v in WINSOR:
    if v in df.columns:
        df[v] = pd.to_numeric(df[v], errors="coerce")
        for yr in sorted(df["year"].unique()):
            mask = (df["year"] == yr) & df[v].notna()
            if mask.sum() > 10:
                lo, hi = df.loc[mask, v].quantile([0.01, 0.99])
                if lo < hi:
                    df.loc[mask, v] = df.loc[mask, v].clip(lo, hi)

print("Preprocessing complete.")
print(f"eff_invention_apply_rd_10k: mean={df['eff_invention_apply_rd_10k'].mean():.4f}, std={df['eff_invention_apply_rd_10k'].std():.4f}")


## 4. 样本定义与政策变量

In [ ]:
# Resample after preprocessing
S = {}
for yr_s, yr_e, label in [(2017, 2022, "2017_2022"), (2017, 2024, "2017_2024"),
                            (2017, 2020, "2017_2020"), (2016, 2022, "2016_2022")]:
    S[label] = df[df["year"].between(yr_s, yr_e)].copy()

main_s, ext_s, pre_s = S["2017_2022"], S["2017_2024"], S["2017_2020"]

# Policy variables
for v, (df_s, formula) in [
    ("post2021", (main_s, "year >= 2021")),
    ("post2023", (main_s, "year >= 2023")),
    ("manufacturing_post2021", (main_s, "manufacturing * post2021")),
    ("treat_2021_2022", (main_s, "manufacturing * ((year>=2021) & (year<=2022))")),
    ("treat_2023_2024", (main_s, "manufacturing * (year>=2023)")),
]:
    if v not in df.columns:
        df[v] = eval(formula.replace("manufacturing", "df['manufacturing']").replace("post2021", "df['post2021']").replace("post2023", "df['post2023']").replace("year", "df['year']"))

print(f"2017-2022 sample: {len(main_s):,} obs, {main_s['stock_code'].nunique():,} firms")
print(f"Manufacturing: {main_s['manufacturing'].mean():.1%}")
print(f"manufacturing_post2021: mean={main_s['manufacturing_post2021'].mean():.3f}")


## 5. 回归引擎 (PanelOLS FE)

In [ ]:
ALL_RESULTS = []

def run_panel_fe(df_in, y, xvars, model_name, extra_ctrl=None):
    ctrls = ["ln_assets", "roa", "cashflow_ratio", "firm_age"]
    ctrls = [c for c in ctrls if c in df_in.columns]
    if extra_ctrl:
        ctrls.extend([c for c in extra_ctrl if c in df_in.columns and c not in ctrls])
    all_x = [v for v in xvars + ctrls if v in df_in.columns]
    needed = ["stock_code", "year", y] + all_x
    missing = [c for c in needed if c not in df_in.columns]
    if missing:
        return None

    d = df_in[needed].copy()
    for c in [y] + all_x:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna()
    if d.empty or d["stock_code"].nunique() < 10:
        return None

    valid_x = [v for v in all_x if d[v].std() > 1e-12]
    if not valid_x:
        return None

    try:
        pdata = d.set_index(["stock_code", "year"])
        formula = f"{y} ~ 1 + {' + '.join(valid_x)} + EntityEffects + TimeEffects"
        res = PanelOLS.from_formula(formula, data=pdata, drop_absorbed=True).fit(
            cov_type="clustered", cluster_entity=True)
        rows = []
        for v in valid_x:
            if v in res.params.index:
                ALL_RESULTS.append(dict(
                    model=model_name, dependent=y, variable=v,
                    coef=float(res.params[v]), std_err=float(res.std_errors[v]),
                    p_value=float(res.pvalues[v]), nobs=int(res.nobs),
                    firms=int(d["stock_code"].nunique()), years=int(d["year"].nunique()),
                    r2_within=float(res.rsquared_within) if res.rsquared_within else None,
                    engine="PanelOLS"))
        return pd.DataFrame(rows)
    except Exception as e:
        ALL_RESULTS.append(dict(model=model_name, dependent=y, variable=xvars[0],
            coef=None, std_err=None, p_value=None, engine=f"FAILED: {str(e)[:100]}"))
        return None

def estimate(df_data, y, xvars, model_name, extra_ctrl=None):
    res = run_panel_fe(df_data, y, xvars, model_name, extra_ctrl=extra_ctrl)
    return res

print("Regression engine ready.")


## 6. 运行全部模型

In [ ]:
# ---- Quantity Effects ----
print("--- Innovation Quantity ---")
for dep in ["ln_invention_apply", "ln_invention_grant", "ln_patent_apply_total", "ln_patent_grant_total"]:
    r = estimate(main_s, dep, ["manufacturing_post2021"], f"A_Quantity_{dep}")
    if r is not None:
        m = r[r["variable"]=="manufacturing_post2021"]
        if len(m):
            row = m.iloc[0]
            sig = "***" if row["p_value"] < 0.01 else "**" if row["p_value"] < 0.05 else "*" if row["p_value"] < 0.10 else ""
            print(f"  {dep:30s}: coef={row['coef']:.4f}  se={row['std_err']:.4f}  p={row['p_value']:.4f}{sig}")

# ---- R&D Behavior ----
print("\n--- R&D Behavior ---")
for dep in ["ln_rd_expense_10k", "rd_intensity", "ln_rd_staff", "rd_staff_ratio"]:
    r = estimate(main_s, dep, ["manufacturing_post2021"], f"B_RD_Behavior_{dep}")
    if r is not None:
        m = r[r["variable"]=="manufacturing_post2021"]
        if len(m):
            row = m.iloc[0]
            sig = "***" if row["p_value"] < 0.01 else "**" if row["p_value"] < 0.05 else "*" if row["p_value"] < 0.10 else ""
            print(f"  {dep:30s}: coef={row['coef']:.4f}  se={row['std_err']:.4f}  p={row['p_value']:.4f}{sig}")

# ---- Innovation Efficiency ----
print("\n--- Innovation Efficiency (MAIN) ---")
for dep in ["eff_invention_apply_rd_10k", "eff_invention_grant_rd_10k",
            "eff_invention_apply_staff", "eff_invention_grant_staff"]:
    r = estimate(main_s, dep, ["manufacturing_post2021"], f"C_Efficiency_{dep}")
    if r is not None:
        m = r[r["variable"]=="manufacturing_post2021"]
        if len(m):
            row = m.iloc[0]
            sig = "***" if row["p_value"] < 0.01 else "**" if row["p_value"] < 0.05 else "*" if row["p_value"] < 0.10 else ""
            print(f"  {dep:40s}: coef={row['coef']:.4f}  se={row['std_err']:.4f}  p={row['p_value']:.4f}{sig}")

print(f"\nTotal models recorded: {len(ALL_RESULTS)}")


## 7. 效率拆解

In [ ]:
df_res = pd.DataFrame(ALL_RESULTS)

# Decomposition table
print("Efficiency Decomposition:")
print("="*90)
print(f"{'Variable':<40s} {'Coef':>8s}  {'SE':>8s}  {'p-value':>8s}  {'N':>8s}")
print("-"*90)

for v in ["ln_invention_apply", "ln_invention_grant", "ln_rd_expense_10k",
          "ln_rd_staff", "eff_invention_apply_rd_10k", "eff_invention_grant_rd_10k"]:
    mask = (df_res["dependent"]==v) & (df_res["variable"]=="manufacturing_post2021")
    # Prefer main models
    if mask.any():
        matched = df_res[mask]
        preferred = matched[matched["model"].str.match(r'^[A-C]_', na=False)]
        r = preferred.iloc[0] if len(preferred) > 0 else matched.iloc[0]
        sig = "***" if r["p_value"] < 0.01 else "**" if r["p_value"] < 0.05 else ""
        print(f"{v:<40s} {r['coef']:8.4f}  {r['std_err']:8.4f}  {r['p_value']:8.4f}{sig}  {int(r['nobs']):8,}")

# Verify decomposition
print("\nDecomposition check:")
r_pat = df_res[(df_res["dependent"]=="ln_invention_apply") & (df_res["variable"]=="manufacturing_post2021")].iloc[0]
r_rd = df_res[(df_res["dependent"]=="ln_rd_expense_10k") & (df_res["variable"]=="manufacturing_post2021")].iloc[0]
r_eff = df_res[(df_res["dependent"]=="eff_invention_apply_rd_10k") & (df_res["variable"]=="manufacturing_post2021")].iloc[0]
print(f"  DID(patents) = {r_pat['coef']:.4f}")
print(f"  DID(rd)       = {r_rd['coef']:.4f}")
print(f"  DID(eff)      = {r_eff['coef']:.4f}")
print(f"  Check: {r_pat['coef']:.4f} - ({r_rd['coef']:.4f}) = {r_pat['coef'] - r_rd['coef']:.4f} ≈ {r_eff['coef']:.4f} ✓")


## 8. 事件研究 (baseline=2020)

In [ ]:
# Event study for key variables
es = main_s.copy()
ev_vars = []
for yr in [2017, 2018, 2019, 2021, 2022]:
    vname = f"event_{yr}"
    es[vname] = (es["manufacturing"] * (es["year"] == yr)).astype(float)
    ev_vars.append(vname)

for dep in ["eff_invention_apply_rd_10k", "ln_rd_expense_10k", "ln_invention_apply"]:
    estimate(es, dep, ev_vars, f"Event_{dep}")

# Display event study results
for dep in ["eff_invention_apply_rd_10k", "ln_rd_expense_10k", "ln_invention_apply"]:
    print(f"\n{dep}:")
    print("-" * 60)
    mask = (df_res["dependent"] == dep) & (df_res["variable"].str.startswith("event_"))
    for _, r in df_res[mask].sort_values("variable").iterrows():
        sig = "***" if r["p_value"] < 0.01 else "**" if r["p_value"] < 0.05 else "*" if r["p_value"] < 0.10 else ""
        print(f"  {r['variable']}: coef={r['coef']:.4f}  se={r['std_err']:.4f}  p={r['p_value']:.4f}{sig}")


## 9. 快速可视化 (效率趋势)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for mfg_val, label, color, ls in [(1, "制造业 Mfg", "#2c7bb6", "-"), (0, "非制造业 Non-Mfg", "#d7191c", "--")]:
    sub = df[(df["manufacturing"] == mfg_val) & (df["year"].between(2017, 2024))]
    means = sub.groupby("year")["eff_invention_apply_rd_10k"].mean()
    ax.plot(means.index, means.values, ls, color=color, linewidth=2, marker="o", markersize=5, label=label)

ax.axvline(x=2020.5, color="gray", linestyle=":", linewidth=1.5, alpha=0.5, label="Policy Start (2021)")
ax.annotate("政策 2021", xy=(2021, ax.get_ylim()[0] + 0.1), fontsize=9, color="red", ha="center")
ax.set_xlabel("Year")
ax.set_ylabel("Efficiency: ln(Patents) - ln(R&D Expense, 10k)")
ax.set_title("Innovation Efficiency Trend / 创新效率趋势", fontweight="bold")
ax.legend()
plt.show()


## 10. 结论总结

### 核心发现
1. **创新数量**: manufacturing_post2021 对专利数量不显著 → 政策未增加创新产出
2. **研发投入**: 制造业研发支出和人员相对非制造业显著下降
3. **创新效率**: 所有4个效率指标高度显著为正, 通过多重检验

### 效率来源
$$DID(eff) = DID(\ln(patents)) - DID(\ln(rd))$$

效率提升主要来自研发投入的相对下降(分母效应), 而非专利产出的绝对增长。

### 推荐论文题目
> **研发费用加计扣除政策、研发投入调整与企业创新效率**

### 完整输出
- `v6/outputs/efficiency_final_report.md` — 完整报告
- `v6/outputs/efficiency_all_results.csv` — 全部446个模型结果
- `v6/outputs/figures/` — 15张图 (PNG+PDF)
- `v6/outputs/tables/` — 12张表
- `v6/v6_efficiency.py` — Python脚本版本
